In [289]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from src import synthetic

In [ ]:
FORECAST_HORIZON = 10
NUM_STORES = 4
NUM_DAYS = 365*4


# Generate sales features

In [291]:
from src.features.lags import make_lags

def make_name(lag: pd.DateOffset, prefix: str) -> str:
    key   = list(lag.kwds.keys())[0]
    value = list(lag.kwds.values())[0]
    return f'{prefix}_{key}_{value}'


def make_lags(target: pd.Series, lags: list[pd.DateOffset]) -> pd.DataFrame:
    """ 
    Create lagged features for a given target series.

    Args:
        target (pd.Series): The target series for which to create lagged features. The index of the series should be a datetime index.
        lags (list[pd.DateOffset]): A list of pandas DateOffset objects representing the lags to create.

    Returns:
        pd.DataFrame: A DataFrame containing the lagged features, with each column named according to the lag applied (e.g., 'lag_days_1', 'lag_days_2', etc.).
    """
    
    # Ensure chronological order of the target series from earliest to latest date
    target = target.sort_index(ascending=True)
    
    x = []
    for lag in lags:
        sales_lag = target.shift(freq=lag)
        sales_lag.name = make_name(lag, prefix='lag')
        x.append(sales_lag)

    x = pd.concat(x, axis=1).reindex(target.index)

    return x


def make_diffs(target: pd.Series, diffs: list[pd.DateOffset], lag: pd.DateOffset) -> pd.DataFrame:
    """ 
    Create differenced features for a given target series.
    This function calculates the difference between the current value and the value at a specified lag (y[t]-y[t-lag]).
    During inference y[t] is unknown, so it's best to use lagged  target values to calculate the difference (y[t-lag]-y[t-2*lag]).
    This is why we use the lag parameter to shift the target series before calculating the difference.

    Args:
        target (pd.Series): The target series for which to create differenced features. The index of the series should be a datetime index.
        diffs (list[pd.DateOffset]): A list of pandas DateOffset objects representing the differences to create.
        lag (pd.DateOffset): A pandas DateOffset object representing the lag to use for calculating the difference.

    Returns:
        pd.DataFrame: A DataFrame containing the differenced features, with each column named according to the 
        difference applied (e.g., 'diff_days_1', 'diff_days_2', etc.).
    """
    
    # Ensure chronological order of the target series from earliest to latest date
    # and shift the target series by the specified lag to align historical values with current values
    target_lagged = target.sort_index(ascending=True).shift(freq=lag) 
    
    x = []
    for diff in diffs:
        # Shift the index along the calendar timeline to align historical dates to the target dates
        # Because names might mismatch due to shift, we align on target.index
        historical_target = target_lagged.shift(freq=diff).reindex(target.index)
        
        # Calculate the difference (Current Target - Historical Target)
        sales_diff = target_lagged - historical_target

        sales_diff.name = make_name(diff, prefix='diff')
        x.append(sales_diff)

    x = pd.concat(x, axis=1).reindex(target.index)
    return x


def make_rolling(target: pd.Series, windows: list[int], func: str, lag: pd.DateOffset) -> pd.DataFrame:
    """ 
    Create rolling features for a given target series. This function calculates the rolling mean for the target series over specified window sizes.
    To avoid data leakage, the target series is shifted by the specified lag before calculating the rolling mean.

    Args:
        target (pd.Series): The target series for which to create rolling features. The index of the series should be a datetime index.
        windows (list[int]): A list of integers representing the window sizes for the rolling calculations.
        func (str): The aggregation function to apply to the rolling window (e.g., 'mean', 'sum', etc.).
        lag (pd.DateOffset): A pandas DateOffset object representing the lag to use for calculating the rolling features.

    Returns:
        pd.DataFrame: A DataFrame containing the rolling features, with each column named according to the window
        size applied (e.g., 'rolling_mean_3', 'rolling_mean_5', etc.).
    """
    
    # Ensure chronological order of the target series from earliest to latest date
    # and shift the target series by the specified lag to align historical values with current values
    target_lagged = target.sort_index(ascending=True).shift(freq=lag) 
    
    x = []
    for window in windows:
        rolling_feature = target_lagged.rolling(window=window).agg(func)
        rolling_feature.name = f'rolling_{func}_{window}'
        x.append(rolling_feature)

    x = pd.concat(x, axis=1).reindex(target.index)

    return x

In [298]:

stores = synthetic.store_data(n_stores=NUM_STORES)
sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)

store1 = sales[sales['Store'] == 1]
sales = sales.loc[store1.index].set_index(['Date'])['Sales'].sort_index(ascending=True) # Sort from earliest to latest date

sales.loc[sales == 0] = np.nan
sales.ffill(inplace=True)  # Fill NaN values with the last valid observation


In [301]:


diffs = [pd.DateOffset(days=i) for i in range(1, 2)]
lags = [pd.DateOffset(days=i) for i in range(1, 2)]
windows = ['7D']
sales_ = sales.sample(frac=1, random_state=42)

xlag = make_lags(sales_, lags)
xdiff = make_diffs(sales_, diffs, lag=pd.DateOffset(days=1))
xroll_mean = make_rolling(sales_, windows, func='mean', lag=pd.DateOffset(days=1))
xroll_std = make_rolling(sales_, windows, func='std', lag=pd.DateOffset(days=1))

gg = pd.concat([sales_, xlag, xdiff, xroll_mean, xroll_std], axis=1).sort_index(ascending=True).head(100)
gg.head(20)

,Sales,lag_days_1,diff_days_1,rolling_mean_7D,rolling_std_7D
Date,,,,,
2013-01-01,467.320508,NaN,NaN,NaN,NaN
2013-01-02,467.820508,467.320508,NaN,467.320508,NaN
2013-01-03,451.000000,467.820508,0.500000,467.570508,0.353553
2013-01-04,434.179492,451.000000,-16.820508,462.047005,9.570253
2013-01-05,534.679492,434.179492,-16.820508,455.080127,15.975275
2013-01-06,534.679492,534.679492,100.500000,471.000000,38.191869
2013-01-07,453.000000,534.679492,0.000000,481.613249,42.927163
2013-01-08,470.820508,453.000000,-81.679492,477.525642,40.651907
2013-01-09,571.320508,470.820508,17.820508,478.025642,40.526800


In [302]:

sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)

sales_grouped = (sales
                 .set_index(['Date'])
                 .sort_index(ascending=True)
                 .groupby('Store')
                 ['Sales'])

x_lag = sales_grouped.apply(lambda x: make_lags(x, lags), include_groups=False)
x_diff = sales_grouped.apply(lambda x: make_diffs(x, diffs, lag=pd.DateOffset(days=1)), include_groups=False)
x_window = sales_grouped.apply(lambda x: make_rolling(x, windows, func='mean', lag=pd.DateOffset(days=1)), include_groups=False)

In [307]:
x_window

rolling_mean_7D
Store Date                       
1     2013-01-01              NaN
      2013-01-02       467.320508
      2013-01-03       467.570508
      2013-01-04       462.047005
      2013-01-05       455.080127
...                           ...
4     2016-12-26      3258.000000
      2016-12-27      3274.285714
      2016-12-28      3276.285714
      2016-12-29      3278.285714
      2016-12-30      3280.285714

[5840 rows x 1 columns]

In [ ]:
# TODO: Calendar features (day of week, month, year, etc.) On the target dates, not the historical dates used for lags/diffs/rolling features!